# ml_model_collection — GPU benchmark on Google Colab

Runs `tools/fetch_model.py` and `tools/benchmark.py` for every object detection
model on the Colab GPU, then downloads the resulting `benchmarks.yaml` files.
Merge them locally with `python tools/merge_benchmarks.py benchmarks_colab.zip`.

1. *Runtime → Change runtime type → GPU* (T4 / L4 / A100).
2. Run all cells. Do not run anything else in the session while benchmarking.

Notes
- Artifacts are exported **inside Colab** from the same pinned upstream sources;
  each record stores the SHA-256 of the exact file measured.
- Exporters for YOLO11n / YOLO26n install `ultralytics` (AGPL-3.0) into this
  throwaway VM only.
- Colab CPUs are shared vCPUs, so CPU records are off by default.

In [ ]:
# --- settings -------------------------------------------------------------
REPO_URL = "https://github.com/rsasaki0109/ml_model_collection"  # empty -> upload zip
# Private repo: add a Colab secret named GITHUB_TOKEN (key icon in the left bar,
# read-only token for this repo) and enable notebook access. The token is only
# used for this clone and is not printed.
HARDWARE_CLASS = "workstation"   # T4/L4/A100 are data-center GPUs (see docs/hardware.md)
RUN_CPU = False
MODELS = []            # empty = all object_detection models

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import subprocess
GPU_NAME = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().splitlines()[0]
HARDWARE_LABEL = f"{GPU_NAME.replace('NVIDIA ', '')} (Colab)"
print(HARDWARE_LABEL)

In [ ]:
import os, zipfile
os.chdir("/content")
if REPO_URL:
    url = REPO_URL
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repository or no secret
    subprocess.run(["git", "clone", "-q", url, "ml_model_collection"], check=True)
    subprocess.run(["git", "-C", "ml_model_collection", "remote", "set-url", "origin", REPO_URL])
else:
    from google.colab import files
    print("Upload dist/ml_model_collection.zip (python tools/pack_for_colab.py)")
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall("/content")
os.chdir("/content/ml_model_collection")
!ls

In [ ]:
# Inference + benchmark deps, and every exporter's deps (one throwaway env).
!pip install -q onnx "onnxruntime-gpu==1.22.0" nvidia-ml-py psutil gdown pyyaml opencv-python-headless     "rfdetr==1.11.0" ultralytics tensorboard "faster-coco-eval>=1.6.5" calflops
import onnxruntime as ort; print(ort.__version__, ort.get_available_providers())

In [ ]:
# Download / export all artifacts (pinned sources, see each model.yaml).
args = " ".join(MODELS) if MODELS else "--task object_detection"
!python tools/fetch_model.py {args}
!python tools/validate.py

In [ ]:
# GPU benchmark. benchmark.py refuses to run if CPU/GPU are busy. Colab VMs idle
# at ~20% CPU (Jupyter/runtime services), so the limit is raised to 40%; the
# measured load is stored in every record.
import time; time.sleep(30)  # let pip/export background work settle
targets = [f"--model {m}" for m in MODELS] or ["--task object_detection"]
for t in targets:
    !python tools/benchmark.py {t} --provider cuda --hardware-label "{HARDWARE_LABEL}" --hardware-class {HARDWARE_CLASS} --max-cpu-load 40
    if RUN_CPU:
        !python tools/benchmark.py {t} --provider cpu --hardware-label "Colab vCPU" --hardware-class cpu_only --iters 60 --warmup 5


In [ ]:
# Package and download the records.
import glob
out = "/content/benchmarks_colab.zip"
with zipfile.ZipFile(out, "w") as z:
    for f in glob.glob("object_detection/*/benchmarks.yaml") + glob.glob("object_detection/*/weights/provenance.json"):
        z.write(f)
print(open(glob.glob("object_detection/*/benchmarks.yaml")[0]).read()[:1500])
from google.colab import files
files.download(out)